# Dockerizing an ML API

---

In this notebook, we will containerize the Iris Prediction API we built in the API Development section. By the end, you'll be able to ship the entire application (model, API, and dependencies) with a single `docker compose up` command.

We will cover:
- The challenges specific to containerizing ML applications.
- Writing a production Dockerfile for our FastAPI + scikit-learn app
- Handling model artifact paths inside a container
- Setting up Docker Compose for one-command workflow
- Testing the containerized API
- Best practices for image size and security.

> ⚠️ **Prerequisite:** Make sure Docker Desktop is installed and running. The Dockerfile and docker-compose.yml live in this folder (`03_containerization/`).

---

## 1. The ML Specific Challenge

Containerizing a regular web app is straightforward: copy code, install dependencies, run. But ML applications have one extra complication: the **model artifact**. 

Our `iris_pipeline.joblib` file lives in `01_model_presistence/models`. The API's `model_loader.py` resolves its path using `Path(__file__)`. Inside a Docker container, the file system is completely different from your laptop, so we need to:

1. **Copy the model artifact into the image** (or mount it as a volume)
2. **Adjust the model path** so it works inside the container.

We'll solve this by copying both the API code and the model into a well-defined directory structure inside the image.

---

## 2. The Dockerfile Explained

Here's the Dockerfile we'll use (it lives at `03_containerization/Dockerfile`):

```dockerfile
# -------- Stage 1: Base image --------
FROM python:3.13-slim

# Prevent Python from writing .pyc files and enable unbuffered output
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONBUFFERED=1

# Set working directory
WORKDIR /app

# -------- Stage 2: Install dependencies --------
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# -------- Stage 3: Copy application code --------
# Copy the FastAPI app
COPY app/ ./app/

# Copy the model artifact
COPY models/iris_pipeline.joblib ./models/iris_pipeline.joblib

# Stage 4: Run --------
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Line-by-Line Walkthrough

| **Line** | **Why** |
| :--- | :--- |
| `FROM python:3.13-slim` | Matches your project's `requires-python = ">=3.13"`. The `-slim` variant is ~150MB smaller than the full image |
| `PYTHONDONTWRITEBYTECODE=1` | Prevents Python from creating `.pyc` files inside the container (keeps the image clean) |
| `PYTHONBUFFERED=1` | Ensures `print()` statements and logs appear immediately in `docker logs` (not buffered). Critical for debugging. |
| `COPY requirements.txt .` -> `RUN pip install`| Dependencies first, code second. This maximizes layer caching. |
| `COPY app/ ./app/` | Copies the FastAPI application (`main.py`, `schemas.py`, `model_loader.py`) |
| `COPY models/...` | Copies the trained model artifact into the image |

---

## 3. The Requirements File

The Docker image doesn't use our `pyproject.toml` or `uv`. It needs a plain requirements.txt with only the packages API actually needs (not our entire data science toolkit).

```
# requirements.txt (for the Docker image only)
fastapi>=0.133.1
uvicorn>=0.41.0
scikit-learn>=1.7.2
numpy>=2.3.4
joblib>=1.5.2
```

Notice what's **not** here: pandas, matplotlib, seaborn, jupyter, xgboost, onnxruntime, etc. The API only needs FastAPI, Uvicorn, scikit-learn (for the Pipeline), and numpy. Keeping the requirements minimal makes the image much smaller.

---

## 4. Adapting the Model Loader

Inside the Docker container, the file structure will look like:

```
/app/
├── app/
│   ├── main.py
│   ├── schemas.py
│   └── model_loader.py
├── models/
│   └── iris_pipeline.joblib
└── requirements.txt
```

Our current `model_loader.py` uses a relative path from `__file__` that navigates to `01_model_persistence/models/`. Inside the container, the model is at `/app/models/` instead.

We'll solve this with an **environment variable** that can be overridden:

In [ ]:
import os
from pathlib import Path

MODEL_PATH = Path(
    os.environ.get(
        "MODEL_PATH", # Default: the path that works when running locally from 02_api_development/
        str(Path(__file__).resolve().parent.parent.parent / "01_model_persistence" / "models" / "iris_pipeline.joblib")
        )
    )

In `docker-compose.yml`, we set the environment variable:

```yaml
environment:
  - MODEL_PATH=/app/models/iris_pipeline.joblib
```

This way the same code works both locally and in Docker without any changes.

---

## 5. Docker Compose Setup

Our `docker-compose.yml` ties everything together:

```yaml
services:
  iris-api:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models/iris_pipeline.joblib
```

| **Key** | **What It Does** |
| :--- | :--- |
| `services.irisi-api` | Defines a service named `iris-api`|
| `build context: .`| Uses the current directory as the build context |
| `build.dockerfile: Dockerfile` | Specifies which Dockerfile to use |
| `ports: "8000:8000"`| Maps host port 8000 to container port 8000. |
| `environment` | Sets the `MODEL_PATH` environment variable inside the container. |

---

## 6. Building and Running

### 6.1. Project Setup

Before building, we need to copy the required files into the `03_containerization/` directory. The Dockerfile's build context is this folder, so it can only access files inside it:

In [ ]:
# From the 03_containerization/ directory:

# Copy the FastAPI app
cp -r ../02_api_development/app ./app

# Create models directory and copy the model artifact
mkdir -p models
cp ../01_model_persistence/models/iris_pipeline.joblib ./models/

### 6.2. Build and Run with Compose

In [ ]:
# Build the image and start the container
docker compose up --build

You should see:

```
iris-api-1  | ✅ Model loaded successfully.
iris-api-1  | INFO:     Uvicorn running on http://0.0.0.0:8000
```

### 6.3. Test It
In a new terminal:

In [ ]:
# Health check
curl http://localhost:8000/health
# {"status":"healthy","model_loaded":true}

# Prediction
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}'
# {"prediction":"setosa","prediction_id":0,"probabilities":{"setosa":0.97,...}}

Or visit `http://localhost:8000/docs` for the Swagger UI.

### 6.4. Stop

In [ ]:
docker compose down


---

## 7. Best Practices for ML Docker Images

### 7.1. Keep Images Small

| **Strategy** | **Why** |
| :--- | :--- |
| Use `-slim` base images | `python:3.13-slim` is ~150MB. `python:3.13` is ~900MB |
| Minimal `requirements.txt` | Only include what the API needs - not your entire data science toolkit |
| `pip install --no-cache-dir` | Prevents pip from storing download cache inside the image. |
| Use `.dockerignore` | Prevent notebooks, `.git/`, `.venv/`, etc. from being copied.|  

### 7.2. Security

| **Strategy** | **Why** |
| :--- | :--- |
| Don't run as root | Add `RUN useradd -m appuser` and `USER appuser` in your Docker file |
| Pin dependency versions | Use exact versions (`scikit-learn==1.7.2`) in production to avoid surprises. |
| Scan images | Use `docker scout quickview` to check for known vulnerabilities |

### 7.3. Model Artifact Strategy

| **Approach** | **Pros** | **Cons** | **Best For** |
| :--- | :--- | :--- | :--- |
| **Copy to image** (what we did) | Simple, self-contained | Image must be rebuilt when model changes | Small models, infrequent updates |
| **Volume mount** | Models can be updated without rebuilding | Requires model file on host | Development, frequent retraining |
| **Download at startup** | Image stays small | Adds startup time, needs network | Large models, cloud storage (S3) |

---

## Summary 

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **ML Docker Challenge** | The model artifact must be accessible inside the container (copy it in, mount it, or download it). |
| `python:3.13-slim` | The right base image: matches our Python version, keeps the image small |
| `PYTHONUNBUFFERED=1` | Ensures logs appear in `docker logs` immediately. Essential for debugging. |
| **Minimal requirements** | Only install what the API needs. Don't ship pandas/matplotlib in production. |
| **Environment variable for paths** | Use `os.environ.get("MODEL_PATH", default)` so the same code works locally and in Docker. |
| `--host 0.0.0.0` | Required for Uvicorn inside Docker. Without it, the container rejects external connections. |
| `docker compose up --build` | One command to build and run the entire application | 

---

**Next section:** [Cloud Deployment](../04_cloud_deployment/) — Putting your containerized API on the internet.